# Reduced Polariton–Soliton Computational Model

A Colab-ready reduced model using a scalar driven-dissipative Gross–Pitaevskii equation with a synthetic hierarchical/helical potential. It is an **inspired reduced model, not a reproduction of a specific experiment**.

\[
i\partial_t\psi=\left[-{
abla^2\over2m}+V+g|\psi|^2+g_Rn_R+{i\over2}(Rn_R-\gamma)ight]\psi,
\qquad
\partial_t n_R=P-(\gamma_R+R|\psi|^2)n_R.
\]

The notebook computes density, phase, reservoir, momentum-space density, transport, spectra, a mean-field \(g^{(2)}\)-like correlation, pump sweeps, and a simple ternary/non-Boolean reservoir-computing task.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import fft2, ifft2, fftfreq
from scipy.ndimage import gaussian_filter
np.random.seed(7)
plt.rcParams["figure.figsize"]=(7,5)

## 1. Grid and parameters

In [ ]:
N=192
L=30.0
dx=L/N
x=(np.arange(N)-N//2)*dx
y=x.copy()
X,Y=np.meshgrid(x,y,indexing="xy")
kx=2*np.pi*fftfreq(N,d=dx)
ky=2*np.pi*fftfreq(N,d=dx)
KX,KY=np.meshgrid(kx,ky,indexing="xy")
K2=KX**2+KY**2

m_eff=1.0
g=0.8
gR=0.6
R=1.2
gamma=0.45
gammaR=0.8
dt=0.012
n_steps=3500
save_every=20
print("grid:",N,"x",N," dx=",dx," total time=",n_steps*dt)

## 2. Synthetic hierarchical/helical potential

\[
V(r,	heta)=V_{m conf}(r)+\sum_\ell A_\ell\cos(k_\ell r+q_\ell	heta+\phi_\ell).
\]

This is phenomenological and does not claim to reproduce exact molecular geometry.

In [ ]:
def make_potential(X,Y,levels=4,seed=3):
    rng=np.random.default_rng(seed)
    r=np.sqrt(X**2+Y**2)+1e-12
    th=np.arctan2(Y,X)
    V=0.10*(r/(0.48*L))**4
    base_k=2*np.pi/8
    for ell in range(levels):
        k=base_k*2**ell
        q=ell+1
        amp=0.18/(1.45**ell)
        ph=rng.uniform(0,2*np.pi)
        V += amp*np.cos(k*r+q*th+ph)
        V += 0.06/(1.5**ell)*np.cos(0.55*k*r-(q+1)*th-ph/2)
    V += -0.25*np.exp(-(r/2)**2)
    d=gaussian_filter(rng.normal(size=X.shape),3)
    d/=np.std(d)
    return V+0.025*d

V=make_potential(X,Y)
plt.figure(); plt.imshow(V,extent=[x.min(),x.max(),y.min(),y.max()])
plt.xlabel("x"); plt.ylabel("y"); plt.title("Synthetic hierarchical/helical potential")
plt.colorbar(label="V"); plt.show()

## 3. Pump profile

In [ ]:
def spot(X,Y,x0,y0,a,s):
    return a*np.exp(-((X-x0)**2+(Y-y0)**2)/(2*s*s))
def make_pump(X,Y,scale=1):
    spots=[(-6,-2.5,1,1.7),(-2,4,.8,1.5),(3.5,2.5,1,1.7),
           (6,-3.5,.75,1.6),(0,-5,.65,1.8)]
    P=np.zeros_like(X)
    for a,b,c,s in spots: P += spot(X,Y,a,b,c*scale,s)
    return P
pump_scale=1.35
P=make_pump(X,Y,pump_scale)
plt.figure(); plt.imshow(P,extent=[x.min(),x.max(),y.min(),y.max()])
plt.title("Pump profile"); plt.colorbar(label="P"); plt.show()

## 4. Split-step ODGPE propagator

In [ ]:
kinetic_half=np.exp(-0.5j*dt*K2/(2*m_eff))
def step(psi,nR,V,P):
    psi=ifft2(fft2(psi)*kinetic_half)
    dens=np.abs(psi)**2
    psi*=np.exp((-1j*(V+g*dens+gR*nR)+0.5*(R*nR-gamma))*dt)
    nR += dt*(P-(gammaR+R*dens)*nR)
    nR=np.maximum(nR,0)
    psi=ifft2(fft2(psi)*kinetic_half)
    return psi,nR

## 5. Run simulation

In [ ]:
rng=np.random.default_rng(10)
psi=.02*(rng.normal(size=(N,N))+1j*rng.normal(size=(N,N)))
nR=np.zeros_like(P)

densities=[]; phases=[]; reservoirs=[]; times=[]; populations=[]; centroids=[]
for s in range(n_steps):
    psi,nR=step(psi,nR,V,P)
    if s%save_every==0:
        d=np.abs(psi)**2
        pop=d.sum()*dx*dx
        xc=(X*d).sum()/(d.sum()+1e-15)
        yc=(Y*d).sum()/(d.sum()+1e-15)
        densities.append(d.astype(np.float32))
        phases.append(np.angle(psi).astype(np.float32))
        reservoirs.append(nR.astype(np.float32))
        times.append(s*dt); populations.append(pop); centroids.append((xc,yc))
densities=np.array(densities); phases=np.array(phases); reservoirs=np.array(reservoirs)
times=np.array(times); populations=np.array(populations); centroids=np.array(centroids)
print("saved frames:",len(times)," final population:",populations[-1])

## 6. Density, phase and reservoir

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(16,4))
for a,data,title in zip(ax,[densities[-1],phases[-1],reservoirs[-1]],
                        [r"Density $|\psi|^2$",r"Phase $\arg(\psi)$",r"Reservoir $n_R$"]):
    im=a.imshow(data,extent=[x.min(),x.max(),y.min(),y.max()])
    a.set_title(title); a.set_xlabel("x"); a.set_ylabel("y"); plt.colorbar(im,ax=a)
plt.tight_layout(); plt.show()

plt.figure(); plt.plot(times,populations)
plt.xlabel("time"); plt.ylabel(r"$\int|\psi|^2dxdy$")
plt.title("Population dynamics"); plt.show()

## 7. Momentum-space density

In [ ]:
nk=np.abs(np.fft.fftshift(fft2(psi)))**2
plt.figure(figsize=(7,6))
plt.imshow(np.log10(nk+1e-12),
           extent=[np.fft.fftshift(kx).min(),np.fft.fftshift(kx).max(),
                   np.fft.fftshift(ky).min(),np.fft.fftshift(ky).max()])
plt.xlabel(r"$k_x$"); plt.ylabel(r"$k_y$")
plt.title(r"Momentum-space $\log_{10}|\psi_k|^2$")
plt.colorbar(); plt.show()

## 8. Centroid transport

In [ ]:
plt.figure()
plt.plot(centroids[:,0],centroids[:,1])
plt.scatter(*centroids[0],label="start"); plt.scatter(*centroids[-1],label="end")
plt.xlabel("x centroid"); plt.ylabel("y centroid"); plt.title("Centroid trajectory"); plt.legend(); plt.show()

v=np.sqrt(np.sum(np.diff(centroids,axis=0)**2,axis=1))/np.diff(times)
plt.figure(); plt.plot(times[1:],v); plt.xlabel("time"); plt.ylabel("centroid speed")
plt.title("Centroid speed"); plt.show()

## 9. Spectrum and \(g^{(2)}\)-like correlation

The spectrum uses total population fluctuations. The correlation is a deterministic mean-field intensity autocorrelation, **not** a quantum-optical \(g^{(2)}\) measurement.

In [ ]:
sig=populations-populations.mean()
sp=np.abs(np.fft.rfft(sig*np.hanning(len(sig))))**2
fr=np.fft.rfftfreq(len(sig),d=times[1]-times[0])
plt.figure(); plt.semilogy(fr[1:],sp[1:]+1e-20); plt.xlim(0,min(3,fr.max()))
plt.xlabel("frequency"); plt.ylabel("power"); plt.title("Population spectrum"); plt.show()

mu=populations.mean()
c=np.correlate(populations-mu,populations-mu,mode="full")[len(populations)-1:]
c/=len(populations)*(mu**2+1e-15)
lag=np.arange(len(c))*(times[1]-times[0])
plt.figure(); plt.plot(lag,1+c); plt.xlim(0,min(10,lag[-1]))
plt.xlabel(r"$\tau$"); plt.ylabel(r"$g^{(2)}$-like"); plt.title("Intensity correlation"); plt.show()

## 10. Pump sweep

In [ ]:
def short_run(scale,steps=1200,seed=1):
    rng=np.random.default_rng(seed)
    ps=.01*(rng.normal(size=(N,N))+1j*rng.normal(size=(N,N)))
    nr=np.zeros_like(P); pp=make_pump(X,Y,scale)
    for _ in range(steps): ps,nr=step(ps,nr,V,pp)
    return np.sum(np.abs(ps)**2)*dx*dx

pump_values=np.linspace(.4,2.0,9)
response=np.array([short_run(p) for p in pump_values])
plt.figure(); plt.plot(pump_values,response,"o-")
plt.xlabel("pump scale"); plt.ylabel("steady-state population")
plt.title("Mean-field pump-response curve"); plt.show()

## 11. Non-Boolean reservoir-computing demonstration

Use \(s_i\in\{-1,0,+1\}\). A generic effective model is

\[
H=-\sum_{ij}J_{ij}s_is_j+\sum_i h_is_i+\sum_iU(s_i).
\]

The next example generates ternary inputs, passes them through nonlinear recurrent dynamics, and trains only a linear readout.

In [ ]:
rng=np.random.default_rng(123)
n_samples,n_in,n_hidden=900,12,36
Xtern=rng.choice([-1,0,1],size=(n_samples,n_in))
w=rng.normal(size=n_in)
J=rng.normal(size=(n_in,n_in)); J=(J+J.T)/2; np.fill_diagonal(J,0)
score=Xtern@w+.35*np.einsum("bi,ij,bj->b",Xtern,J,Xtern)
labels=(score>0).astype(float)

Win=rng.normal(scale=.7,size=(n_hidden,n_in))
Wrec=rng.normal(scale=.12,size=(n_hidden,n_hidden))
states=np.zeros((n_samples,n_hidden))
for b in range(n_samples):
    h=np.zeros(n_hidden)
    for _ in range(8): h=np.tanh(Win@Xtern[b]+Wrec@h)
    states[b]=h

split=int(.8*n_samples)
coef=np.linalg.pinv(np.c_[states[:split],np.ones(split)])@labels[:split]
pred=(np.c_[states[split:],np.ones(n_samples-split)]@coef>.5).astype(float)
print("test accuracy:",np.mean(pred==labels[split:]))

fig,ax=plt.subplots(1,2,figsize=(12,4))
ax[0].imshow(Xtern[:30],aspect="auto",interpolation="nearest")
ax[0].set_title("Ternary input patterns")
ax[1].plot(labels[split:split+80],"o-",label="target")
ax[1].plot(pred[:80],"x--",label="prediction")
ax[1].legend(); ax[1].set_title("Test classification")
plt.tight_layout(); plt.show()

## 12. Physical-field readout

For an actual reservoir experiment, use detector regions

\[
z_j(t)=\int_{R_j}|\psi(x,y,t)|^2\,dxdy
\]

as the physical state supplied to a simple readout.

In [ ]:
regions=[(-8,-2,-7,1),(-3,3,2,7),(2,7,-1,4),(4,8,-6,-1),(-2,2,-7,-3),(-1,3,-1,3)]
features=[]
for d in densities:
    row=[]
    for xa,xb,ya,yb in regions:
        mask=(X>=xa)&(X<xb)&(Y>=ya)&(Y<yb)
        row.append(np.mean(d[mask]))
    features.append(row)
features=np.array(features)

plt.figure()
plt.imshow(features.T,aspect="auto",interpolation="nearest")
plt.xlabel("time frame"); plt.ylabel("readout region")
plt.title("Physical-state feature map"); plt.colorbar(label="mean density"); plt.show()

# 13. What this model establishes

It tests a **reduced nonlinear physical substrate**:

\[
\text{structured potential}
\rightarrow
\text{driven nonlinear field}
\rightarrow
\text{localized/transport dynamics}
\rightarrow
\text{physical reservoir}
\rightarrow
\text{non-Boolean computation}.
\]

It does **not** by itself establish experimental strong coupling, a quantum condensate, a quantum \(g^{(2)}\), a topological invariant, a qubit, entanglement, quantum advantage, or a quantum computer.

### Next research-grade upgrades

1. spinor polaritons \((\psi_+,\psi_-)^T\);
2. TE–TM/spin–orbit coupling;
3. Bogoliubov stability analysis;
4. field-level soliton verification;
5. explicit Thouless-pump/Berry-curvature calculation;
6. stochastic/truncated-Wigner quantum dynamics;
7. a genuine two-dimensional qubit encoding and gates;
8. PyTorch surrogate modeling and ML inverse design.

The safest thesis progression is

\[
\boxed{
\text{ODGPE}
\rightarrow
\text{spinor dynamics}
\rightarrow
\text{soliton stability}
\rightarrow
\text{topological transport}
\rightarrow
\text{reservoir computing}
\rightarrow
\text{ML inverse design}
}
\]

with each arrow treated as a separate falsifiable scientific claim.